In [2]:
import sys, json
sys.path.append("..")
from functools import partial
from src.data import load_cuad
from src.split import split_contracts
from src.chunking import clause_chunks
from src.retrieval import make_dense_ranker
from src.bm25 import make_bm25_ranker
from src.hybrid import make_hybrid_ranker
from src.llm import complete
from src.generate import review_clause

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)
chunker = partial(clause_chunks, max_size=1500)
dense = make_dense_ranker(chunker, tag="clause-200-1500-200")
bm25 = make_bm25_ranker(chunker)
hybrid = make_hybrid_ranker([dense, bm25], weights=[1.0, 2.0])
print("ready")

ready


In [3]:
i = next(i for i in dev_idx if contracts[i]["gold"]["Governing Law"])
c = contracts[i]
passages = hybrid(c, "Governing Law")[:3]

finding = review_clause(c["text"], "Governing Law", passages, complete)
print(c["title"])
print(json.dumps(finding.model_dump(), indent=2) if finding else "COULD NOT VERIFY, review manually")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
{
  "clause": "Governing Law",
  "found": true,
  "risk_level": "Medium",
  "explanation": "The agreement is governed by the laws of the State of Illinois, which may be the provider's home state rather than the customer's jurisdiction. Additionally, the clause does not specify a neutral forum or the customer's home jurisdiction for resolving disputes.",
  "cited_text": "6.9      Governing Law.  This Agreement is to be construed according to the laws          of the State of Illinois.",
  "suggested_redline": "This Agreement is governed by the laws of [Customer's jurisdiction]. The parties submit to the non-exclusive jurisdiction of the courts located in [Customer's jurisdiction].",
  "citation_start": 52036,
  "citation_end": 52151
}


In [4]:
absent = [i for i in dev_idx if not contracts[i]["gold"]["Governing Law"]][:3]

for i in absent:
    c = contracts[i]
    passages = hybrid(c, "Governing Law")[:3]
    f = review_clause(c["text"], "Governing Law", passages, complete)
    print(c["title"])
    if f is None:
        print("  COULD NOT VERIFY")
    elif not f.found:
        print("  found: False  (good)")
    else:
        print("  found: True  | risk:", f.risk_level)
        print("  quote:", f.cited_text[:200])
    print("-" * 60)

NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT
  found: False  (good)
------------------------------------------------------------
ThriventVariableInsuranceAccountB_20190701_N-6_EX-99.D(IV)_11720968_EX-99.D(IV)_Endorsement Agreement
  found: False  (good)
------------------------------------------------------------
Freecook_20180605_S-1_EX-10.3_11233807_EX-10.3_Hosting Agreement
  found: False  (good)
------------------------------------------------------------
